In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from processing.spectrometer import read_folder

base_root = "/Users/rodionsa/KAUST Dropbox/Sergei Rodionov/Lab_measurements_shared/HoloEML/HoloEML281026/spectrometer/"

In [ ]:
# TiO2 sample

# R
s_r = read_folder(base_root + "TiO2/R", smooth=30, grid=(430, 900, 5),method="mean")
s_t = read_folder(base_root + "TiO2/T", smooth=90, grid=(430, 900, 5), method="mean")

plt.plot(s_r.wavelength, s_r['square1'])
plt.plot(s_t.wavelength, s_t['square2'])
plt.plot(s_t.wavelength, s_t['square2']+s_r['square2'])
plt.ylim([0, 110])
plt.grid()

In [ ]:
# Fit data
from dispertorch import Stack, ConstantEps, TaucLorentz, Measurement, fit, material, list_materials, CauchyUrbach
from ridb import RIDB
from processing.references import mirror_reflectance

wvl = torch.linspace(430, 900, 95)

# TiO2 from ridb
ridb = RIDB()
TiO2_old = ridb.get_material("TiO2_ebeam_am_030925")

# Create materials
air = material("air")
SiO2 = material('SiO2')

P = lambda v: nn.Parameter(torch.as_tensor(v, dtype=torch.float32))
# Starting values inside the bounds below (fit clamps anything outside to the edge).
# k_exp fixed: the data cannot determine it (R+T fit pushes it to 0, i.e. flat k).
TiO2 = CauchyUrbach(A=P(2.1), B=P(0.03), k_amp=P(0.05), k_exp=1.0, E_band=2.5)
#TaucLorentz(A=P(150.0), omega_0=P(4.5), gamma=P(1.5), E_g=P(3.2))

cu_bounds = {
    "A":     (1.8, 2.6),     # n at long wavelength (TiO2 ~2.0–2.5 in the visible)
    "B":     (0.0, 0.1),     # µm²; >= 0 means n rises toward the blue (normal dispersion)
    "k_amp": (0.0, 0.5),     # k at E_band; must be >= 0
}

# Fit TiO2
TiO2_n_meas = Measurement(kind="n", wl=wvl, data=TiO2_old.n(wvl).real)
TiO2_k_meas = Measurement(kind="k", wl=wvl, data=TiO2_old.n(wvl).imag)

fit(TiO2, [TiO2_n_meas, TiO2_k_meas], method='lm', max_iter=100, tol=1e-5, bounds=cu_bounds)

# Reference correction: files hold R_sample / R_mirror and T_sample / T_bare_glass.
# Multiply back by the reference curves to get absolute R and T (fractions).
# Mirror: Thorlabs protected silver (-P01) datasheet, 12 deg AOI, unpolarized.
d_sub = 500000000  # substrate thickness, nm; same for sample stack and glass reference
R_mirror = mirror_reflectance(s_r.wavelength)
with torch.no_grad():
    T_glass = Stack(air, [], SiO2, substrate_thickness=d_sub, exit_medium=air)(wvl).T.numpy()

R_abs = s_r['square1'] / 100 * R_mirror
T_abs = s_t['square2'] / 100 * T_glass

# R below ~500 nm carries the 50x objective's back-reflection (not subtracted as dark):
# up to +7% at 430 nm vs the T-only prediction. Fit R only above R_cut; T on the full range.
R_cut = 500  # nm
R_sel = (wvl >= R_cut).numpy()

# Fit model
stack = Stack(air, [(TiO2, P(200.0))], SiO2, substrate_thickness=d_sub, exit_medium=air)
R_meas = Measurement(kind="R", wl=wvl[R_sel], data=R_abs[R_sel])
T_meas = Measurement(kind="T", wl=wvl, data=T_abs)

# Same material bounds, renamed for their place in the stack, plus film thickness (nm).
stack_bounds = {f"layers.0.model.{k}": v for k, v in cu_bounds.items()}
stack_bounds["layers.0.d"] = (100.0, 300.0)

# Always fit R together with T: R alone falls into an absorbing ~178 nm solution.
res = fit(stack, [R_meas, T_meas], method='lm', max_iter=100, tol=1e-5, bounds=stack_bounds)

print(res.summary())

In [ ]:
with torch.no_grad():
    out = stack(wvl)

plt.plot(wvl, out.R, label="R model")
plt.plot(s_r.wavelength, R_abs, ".", ms=3, label="R measured (× R_mirror)")

plt.plot(wvl, out.T, label="T model")
plt.plot(s_t.wavelength, T_abs, ".", ms=3, label="T measured (× T_glass)")

plt.axvspan(wvl[0], R_cut, color="0.9", zorder=0, label="R excluded from fit")
plt.xlabel("Wavelength (nm)")
plt.legend()
plt.grid()

In [ ]:
with torch.no_grad():
    plt.plot(wvl, stack.layers[0].model.n(wvl))

In [ ]:
def nk_bands(res, system, model, wl):
    """1-sigma bands on n(wl), k(wl) from the fit covariance (linearized)."""
    err = torch.cat([res.stderr[n].reshape(-1) for n in res.params]).double()
    cov = res.correlation.double() * torch.outer(err, err)
    named = dict(system.named_parameters())
    params = [named[n] for n in res.params]
    n, k = model.nk(wl)
    rows = []
    for y in torch.cat([n, k]):
        g = torch.autograd.grad(y, params, retain_graph=True, allow_unused=True)
        rows.append(torch.cat([(torch.zeros_like(p) if gi is None else gi).reshape(-1)
                               for gi, p in zip(g, params)]))
    J = torch.stack(rows).double()
    sig = torch.sqrt(torch.clamp(torch.einsum("ip,pq,iq->i", J, cov, J), min=0)).float()
    m = len(wl)
    return n.detach(), k.detach(), sig[:m], sig[m:]

n, k, sn, sk = nk_bands(res, stack, TiO2, wvl)

In [ ]:
plt.plot(sk/k)

In [ ]:
res.correlation